# Deploy a Nova Model to Amazon Bedrock

This notebook demonstrates how to fine-tune an Amazon Nova model using the SageMaker SDK
and deploy it to Amazon Bedrock using `BedrockModelBuilder`.

The workflow:
1. Fine-tune Nova Micro using `SFTTrainer`
2. Create a `BedrockModelBuilder` from the completed training job
3. Deploy to Bedrock — the builder automatically:
   - Detects the model as Nova
   - Reads the checkpoint URI from the training job manifest
   - Calls `CreateCustomModel`
   - Polls until the model is Active
   - Calls `CreateCustomModelDeployment`
   - Polls until the deployment is Active
4. Clean up resources

**Prerequisites:**
- AWS credentials with SageMaker and Bedrock access in `us-east-1`
- `sagemaker-serve` and `sagemaker-train` packages installed
- An IAM role with Bedrock and SageMaker permissions

## Setup

In [ ]:
import os
import json
import time
import random
import boto3

REGION = "us-east-1"
os.environ["AWS_DEFAULT_REGION"] = REGION

from sagemaker.core.helper.session_helper import get_execution_role

role_arn = get_execution_role()
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{REGION}-{account_id}"

print(f"Region:  {REGION}")
print(f"Account: {account_id}")
print(f"Role:    {role_arn}")
print(f"Bucket:  {bucket}")

## Step 1: Prepare training data

Upload a small JSONL dataset in the chat-messages format that Nova expects.

In [ ]:
s3 = boto3.client("s3", region_name=REGION)

train_key = "nova-example/train.jsonl"
train_uri = f"s3://{bucket}/{train_key}"

rows = []
for i in range(50):
    rows.append(json.dumps({
        "messages": [
            {"role": "user", "content": f"What is {i+1} + {i+1}?"},
            {"role": "assistant", "content": f"The answer is {(i+1)*2}."}
        ]
    }))

s3.put_object(Bucket=bucket, Key=train_key, Body="\n".join(rows).encode())
print(f"Uploaded {len(rows)} examples to {train_uri}")

## Step 2: Fine-tune Nova Micro with SFTTrainer

This launches a SageMaker training job. It typically takes 15-30 minutes to complete.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer

trainer = SFTTrainer(
    model="nova-textgeneration-micro",
    training_dataset=train_uri,
    accept_eula=True,
    model_package_group="nova-example-models",
)

# Set wait=True to block until training completes
trainer.train(wait=True)

training_job = trainer._latest_training_job
print(f"Training job: {training_job.training_job_name}")
print(f"Status:       {training_job.training_job_status}")

## Step 3: Deploy to Bedrock with BedrockModelBuilder

The builder handles the full deployment flow:
- Fetches the model package from the training job
- Detects it as a Nova model
- Reads the checkpoint URI from the training output manifest
- Creates a Bedrock custom model and polls until Active
- Creates a deployment and polls until Active

In [ ]:
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder

builder = BedrockModelBuilder(model=training_job)

print(f"Model package:    {builder.model_package}")
print(f"S3 artifacts:     {builder.s3_model_artifacts}")

In [ ]:
rand = random.randint(1000, 9999)
custom_model_name = f"nova-example-{rand}-{int(time.time())}"
deployment_name = f"{custom_model_name}-dep"

print(f"Deploying as: {custom_model_name}")
print(f"This will poll for model creation and deployment — may take several minutes...")

response = builder.deploy(
    custom_model_name=custom_model_name,
    role_arn=role_arn,
    deployment_name=deployment_name,
)

deployment_arn = response.get("customModelDeploymentArn")
print(f"\nDeployment ARN: {deployment_arn}")
print("Deployment is Active and ready for inference.")

## Step 4: Test inference (optional)

Once the deployment is Active, you can invoke it via the Bedrock Runtime API.

In [ ]:
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

# Get the model ARN from the deployment
bedrock = boto3.client("bedrock", region_name=REGION)
dep_info = bedrock.get_custom_model_deployment(
    customModelDeploymentIdentifier=deployment_arn
)
model_arn = dep_info.get("modelArn")
print(f"Model ARN: {model_arn}")

# Invoke
invoke_response = bedrock_runtime.invoke_model(
    modelId=deployment_arn,
    contentType="application/json",
    body=json.dumps({
        "messages": [{"role": "user", "content": "What is 7 + 7?"}]
    }),
)

result = json.loads(invoke_response["body"].read())
print(f"Response: {result}")

## Step 5: Cleanup

Delete the deployment and custom model to avoid ongoing charges.

In [ ]:
bedrock = boto3.client("bedrock", region_name=REGION)

# Delete deployment first
if deployment_arn:
    try:
        bedrock.delete_custom_model_deployment(
            customModelDeploymentIdentifier=deployment_arn
        )
        print(f"Deleted deployment: {deployment_arn}")
    except Exception as e:
        print(f"Failed to delete deployment: {e}")

# Then delete the custom model
if model_arn:
    try:
        bedrock.delete_custom_model(modelIdentifier=model_arn)
        print(f"Deleted custom model: {model_arn}")
    except Exception as e:
        print(f"Failed to delete custom model: {e}")